# Sports Betting Agent

The goal of this notebook is to create an agent that can give you explanation and reasoning on the high confidence bets for a given week.

The first run through code will be a demo w/ hard coded values

In [24]:
# ============================================================
# setup.py - Run this once to set up your environment
# ============================================================

import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

# Verify they loaded
print(f"✓ API Key loaded: {ANTHROPIC_API_KEY[:20]}...")
print(f"✓ GitHub Token loaded: {GITHUB_TOKEN[:20]}...")

✓ API Key loaded: sk-ant-api03-QbLMDE_...
✓ GitHub Token loaded: ghp_SrIAMizvgosgOztf...


In [53]:
from typing import Any
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent
from llama_index.llms.anthropic import Anthropic

# Initialize Claude LLM
llm = Anthropic(api_key=ANTHROPIC_API_KEY, model="anthropic.claude-3-sonnet-20240229-v1:0")
print("✓ Claude LLM initialized")

✓ Claude LLM initialized


hard coded values 

In [54]:
# ============================================================
# Part 1: Define Tool #1 - Model Predictions
# ============================================================

def get_model_predictions(week: int, season: int = 2025) -> dict:
    """
    Fetch XGBoost model predictions for the given NFL week.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with game predictions and confidence scores
    """
    return {
        "week": week,
        "games": [
            {
                "game_id": "KC-LV",
                "matchup": "Chiefs vs Raiders",
                "spread": -7.5,
                "model_prediction": "Chiefs -7.2",
                "confidence": 0.68,
                "ats_probability": 0.65,
                "edge": 0.042
            },
            {
                "game_id": "BUF-MIA",
                "matchup": "Bills vs Dolphins",
                "spread": -6.0,
                "model_prediction": "Bills -5.8",
                "confidence": 0.62,
                "ats_probability": 0.58,
                "edge": 0.018
            }
        ]
    }

# Convert to LlamaIndex tool
prediction_tool = FunctionTool.from_defaults(get_model_predictions)

print("✓ Prediction tool created")

# Test it
test_result = get_model_predictions(week=1)
print(f"\nTest prediction tool:")
print(f"Games: {len(test_result['games'])} games")
print(f"First game: {test_result['games'][0]['matchup']}")

✓ Prediction tool created

Test prediction tool:
Games: 2 games
First game: Chiefs vs Raiders


In [55]:
# ============================================================
# Part 2: Define Tool #2 - Injury Reports
# ============================================================

def get_injury_reports(week: int, season: int = 2025) -> dict:
    """
    Fetch current NFL injury reports for the week.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with injured players by team and their impact level
    """
    return {
        "week": week,
        "injuries": [
            {
                "team": "KC",
                "player": "Chris Jones",
                "position": "DE",
                "severity": "out",
                "impact": "high",
                "estimated_return": "4 weeks"
            },
            {
                "team": "BUF",
                "player": "Damar Hamlin",
                "position": "S",
                "severity": "questionable",
                "impact": "medium",
                "estimated_return": "2 weeks"
            },
            {
                "team": "LV",
                "player": "Chandler Jones",
                "position": "DE",
                "severity": "out",
                "impact": "high",
                "estimated_return": "6 weeks"
            }
        ]
    }

# Convert to LlamaIndex tool
injury_tool = FunctionTool.from_defaults(get_injury_reports)

print("✓ Injury reports tool created")

# Test it
test_injuries = get_injury_reports(week=1)
print(f"\nTest injury tool:")
print(f"Total injured players: {len(test_injuries['injuries'])}")
for injury in test_injuries['injuries']:
    print(f"  - {injury['team']}: {injury['player']} ({injury['position']}) - {injury['impact']} impact")

✓ Injury reports tool created

Test injury tool:
Total injured players: 3
  - KC: Chris Jones (DE) - high impact
  - BUF: Damar Hamlin (S) - medium impact
  - LV: Chandler Jones (DE) - high impact


In [56]:
# ============================================================
# Part 3: Define Tool #3 - Line Movement
# ============================================================

def get_line_movement(week: int, season: int = 2025) -> dict:
    """
    Fetch Vegas line movement data for the week.
    Shows opening line, current line, and public vs sharp action.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with line movement data for each game
    """
    return {
        "week": week,
        "games": [
            {
                "game_id": "KC-LV",
                "opening_line": -7.0,
                "current_line": -7.5,
                "movement": -0.5,
                "public_percentage": 0.68,
                "sharp_percentage": 0.55,
                "notes": "Public heavily backing Chiefs, sharp money fading slightly"
            },
            {
                "game_id": "BUF-MIA",
                "opening_line": -5.5,
                "current_line": -6.0,
                "movement": -0.5,
                "public_percentage": 0.72,
                "sharp_percentage": 0.48,
                "notes": "Heavy public action on Bills, sharp money on Dolphins"
            }
        ]
    }

# Convert to LlamaIndex tool
line_tool = FunctionTool.from_defaults(get_line_movement)

print("✓ Line movement tool created")

# Test it
test_lines = get_line_movement(week=1)
print(f"\nTest line movement tool:")
print(f"Games with line data: {len(test_lines['games'])}")
for game in test_lines['games']:
    print(f"\n  {game['game_id']}:")
    print(f"    Opening: {game['opening_line']}")
    print(f"    Current: {game['current_line']}")
    print(f"    Movement: {game['movement']}")
    print(f"    Public backing: {game['public_percentage']*100:.0f}%")
    print(f"    Sharp backing: {game['sharp_percentage']*100:.0f}%")
    print(f"    Notes: {game['notes']}")

✓ Line movement tool created

Test line movement tool:
Games with line data: 2

  KC-LV:
    Opening: -7.0
    Current: -7.5
    Movement: -0.5
    Public backing: 68%
    Sharp backing: 55%
    Notes: Public heavily backing Chiefs, sharp money fading slightly

  BUF-MIA:
    Opening: -5.5
    Current: -6.0
    Movement: -0.5
    Public backing: 72%
    Sharp backing: 48%
    Notes: Heavy public action on Bills, sharp money on Dolphins


In [57]:
# ============================================================
# Part 4: Define Tool #4 - Historical Matchup Data (FIXED)
# ============================================================

def get_historical_matchup(team1: str, team2: str) -> dict:
    """
    Get head-to-head historical data between two teams.
    Shows recent performance, trends, and records.
    
    Args:
        team1: First team abbreviation (e.g., "KC")
        team2: Second team abbreviation (e.g., "LV")
    
    Returns:
        Dictionary with head-to-head record and recent game history
    """
    
    # Database of matchup histories (hardcoded for now)
    matchups = {
        ("KC", "LV"): {
            "head_to_head_record": "KC leads 12-5",
            "avg_margin": 3.2,
            "last_5_games": [
                {"date": "2024-12-15", "winner": "KC", "margin": 7, "team1_score": 24, "team2_score": 17},
                {"date": "2024-09-10", "winner": "LV", "margin": 3, "team1_score": 20, "team2_score": 23},
                {"date": "2023-12-24", "winner": "KC", "margin": 10, "team1_score": 31, "team2_score": 21},
                {"date": "2023-09-03", "winner": "KC", "margin": 5, "team1_score": 25, "team2_score": 20},
                {"date": "2022-12-25", "winner": "KC", "margin": 8, "team1_score": 28, "team2_score": 20}
            ],
            "notes": "KC dominates this matchup historically. Has won 4 of last 5 games."
        },
        ("BUF", "MIA"): {
            "head_to_head_record": "BUF leads 7-10",
            "avg_margin": 1.8,
            "last_5_games": [
                {"date": "2024-12-29", "winner": "MIA", "margin": 2, "team1_score": 20, "team2_score": 22},
                {"date": "2024-11-17", "winner": "BUF", "margin": 4, "team1_score": 23, "team2_score": 19},
                {"date": "2024-09-08", "winner": "MIA", "margin": 1, "team1_score": 31, "team2_score": 32},
                {"date": "2023-12-31", "winner": "BUF", "margin": 6, "team1_score": 32, "team2_score": 26},
                {"date": "2023-09-10", "winner": "MIA", "margin": 3, "team1_score": 24, "team2_score": 27}
            ],
            "notes": "Close matchups. Slight MIA edge in recent games but BUF historically favored."
        }
    }
    
    # Normalize team order to handle both (KC, LV) and (LV, KC)
    key = (team1, team2) if (team1, team2) in matchups else (team2, team1)
    
    if key not in matchups:
        # Return default if matchup not in database
        return {
            "team1": team1,
            "team2": team2,
            "head_to_head_record": "No recent history",
            "avg_margin": 0,
            "last_5_games": [],
            "notes": f"No historical data available for {team1} vs {team2}"
        }
    
    matchup_data = matchups[key]
    
    return {
        "team1": team1,
        "team2": team2,
        "head_to_head_record": matchup_data["head_to_head_record"],
        "avg_margin": matchup_data["avg_margin"],
        "last_5_games": matchup_data["last_5_games"],
        "notes": matchup_data["notes"]
    }

# Convert to LlamaIndex tool
history_tool = FunctionTool.from_defaults(get_historical_matchup)

print("✓ Historical matchup tool created")

# Test it with multiple matchups
print("\nTest 1: KC vs LV")
test1 = get_historical_matchup("KC", "LV")
print(f"Record: {test1['head_to_head_record']}")

print("\nTest 2: BUF vs MIA")
test2 = get_historical_matchup("BUF", "MIA")
print(f"Record: {test2['head_to_head_record']}")

✓ Historical matchup tool created

Test 1: KC vs LV
Record: KC leads 12-5

Test 2: BUF vs MIA
Record: BUF leads 7-10


In [58]:
# ============================================================
# Part 5: Define Tool #5 - Model Confidence Analysis
# ============================================================

def analyze_model_confidence(week: int, season: int = 2025) -> dict:
    """
    Analyze overall model confidence for the given week.
    Returns statistics about prediction uncertainty and conviction level.
    
    Args:
        week: NFL week number (1-18)
        season: NFL season year
    
    Returns:
        Dictionary with confidence statistics and recommendations
    """
    return {
        "week": week,
        "average_confidence": 0.617,
        "confidence_distribution": {
            "high_confidence": 5,     # Games with >65% confidence
            "medium_confidence": 8,   # Games with 55-65% confidence
            "low_confidence": 3       # Games with <55% confidence
        },
        "volatility": 0.052,
        "edge_distribution": {
            "high_edge": 2,           # Games with >3% edge
            "medium_edge": 6,         # Games with 1-3% edge
            "low_edge": 8             # Games with <1% edge
        },
        "recommendation": "Medium conviction week - average confidence 61.7%. Consider medium bet sizing.",
        "highest_edge_games": [
            {
                "game": "KC-LV",
                "edge": 0.042,
                "confidence": 0.68,
                "action": "Bet this"
            },
            {
                "game": "BUF-MIA",
                "edge": 0.018,
                "confidence": 0.62,
                "action": "Consider betting"
            }
        ],
        "lowest_confidence_games": [
            {
            }
        ]
    }

# Convert to LlamaIndex tool
confidence_tool = FunctionTool.from_defaults(analyze_model_confidence)

print("✓ Model confidence tool created")

# Test it
test_confidence = analyze_model_confidence(week=1)
print(f"\nTest model confidence tool:")
print(f"Week: {test_confidence['week']}")
print(f"Average confidence: {test_confidence['average_confidence']*100:.1f}%")
print(f"Volatility: {test_confidence['volatility']}")
print(f"\nConfidence distribution:")
print(f"  High (>65%): {test_confidence['confidence_distribution']['high_confidence']} games")
print(f"  Medium (55-65%): {test_confidence['confidence_distribution']['medium_confidence']} games")
print(f"  Low (<55%): {test_confidence['confidence_distribution']['low_confidence']} games")
print(f"\nEdge distribution:")
print(f"  High edge (>3%): {test_confidence['edge_distribution']['high_edge']} games")
print(f"  Medium edge (1-3%): {test_confidence['edge_distribution']['medium_edge']} games")
print(f"  Low edge (<1%): {test_confidence['edge_distribution']['low_edge']} games")
print(f"\nHighest edge games:")
for game in test_confidence['highest_edge_games']:
    print(f"  {game['game']}: {game['edge']*100:.1f}% edge, {game['confidence']*100:.0f}% confidence")
print(f"\nRecommendation: {test_confidence['recommendation']}")

✓ Model confidence tool created

Test model confidence tool:
Week: 1
Average confidence: 61.7%
Volatility: 0.052

Confidence distribution:
  High (>65%): 5 games
  Medium (55-65%): 8 games
  Low (<55%): 3 games

Edge distribution:
  High edge (>3%): 2 games
  Medium edge (1-3%): 6 games
  Low edge (<1%): 8 games

Highest edge games:
  KC-LV: 4.2% edge, 68% confidence
  BUF-MIA: 1.8% edge, 62% confidence

Recommendation: Medium conviction week - average confidence 61.7%. Consider medium bet sizing.


In [59]:
# ============================================================
# Part 6: Combine All Tools & Create Agent (FIXED)
# ============================================================

# Step 1: Collect all tools into a list
tools = [
    prediction_tool,
    injury_tool,
    line_tool,
    history_tool,
    confidence_tool
]

print(f"✓ All {len(tools)} tools registered")

agent = ReActAgent(tools=tools, llm=llm, verbose=True, max_iterations=10,
    system_prompt="""You are an expert sports betting analyst with deep knowledge of NFL games.

    Your job is to analyze NFL games and recommend high-conviction bets.

    When analyzing games, you MUST:
    1. First get model predictions for the week
    2. Check injury reports for key players
    3. Review line movement to see smart money vs public
    4. Look up historical matchups between teams
    5. Analyze overall model confidence for the week

    Then synthesize all this information and:
    - Rank games by confidence and edge
    - Explain why you recommend each bet
    - Flag any conflicts (e.g., model says yes but sharp money says no)
    - Give an overall conviction level for the week
    - Recommend bet sizing based on confidence

    Be conservative with low-confidence predictions. Always explain your reasoning."""
)

print("✓ ReActAgent created with system prompt")
print("\nAgent is ready to analyze games!")

✓ All 5 tools registered
✓ ReActAgent created with system prompt

Agent is ready to analyze games!


In [60]:
# ============================================================
# Part 7: Run the Agent
# ============================================================
from llama_index.core.agent.workflow import AgentStream

handler = agent.run("Analyze week 1 NFL games. Which games should we bet on? "
    "Consider model predictions, injuries, line movement, and historical matchups. "
    "Rank your recommendations by confidence and explain your reasoning for each.")

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler

print("\n" + "="*80)
print("AGENT ANALYSIS - WEEK 1")
print("="*80)
print(final_response)

[tick] add: AgentWorkflowStartEvent(user_msg='Analyze week 1 NFL games. Which games should we be...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Analyze week 1 NFL games. Which games should we bet on? Consider mode...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are an expert sports betting analyst with deep knowledge of N...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with no result


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Caaf2aF6vvPJ8nVattDr8'}

In [ ]:
print(response)